# Evidence contract publisher

Packages one case into a bounded JSON envelope for the Foundry consultation agent.
This is the boundary: the agent never queries the lakehouse, it receives what the
pipeline chose to send.

| | |
| --- | --- |
| **Reads** | the three `gold_case_*` tables, plus the release stamp |
| **Writes** | `Files/agent-handoff/<run_id>/<case_id>/case-evidence.json` |

Every value the agent may state carries an `evidenceId`. Nothing else crosses — no raw
ClinVar JSON, no reference table dumps, no free-text the model could mine for
unsupported claims.

Validation runs before **and** after the write, and `_SUCCESS` is written last. If the
gold tables disagree with the envelope, nothing publishes.

In [ ]:
PIPELINE_RUN_ID = ""
CASE_ID = ""

In [ ]:
import json
import os
from datetime import datetime, timezone

from pyspark.sql import functions as F

# --- cross-lakehouse reads -------------------------------------------------------
# spark.read.table() only resolves against this notebook's default lakehouse, so any
# table in a different layer is read by explicit OneLake path.
_WS = notebookutils.runtime.context["currentWorkspaceId"]
_ONELAKE = notebookutils.conf.get("trident.onelake.endpoint").replace("https://", "")
_LAKEHOUSE_ID = {}


def lake_table(lakehouse, table_name):
    if lakehouse not in _LAKEHOUSE_ID:
        _LAKEHOUSE_ID[lakehouse] = notebookutils.lakehouse.get(
            lakehouse, workspaceId=_WS).id
    path = (f"abfss://{_WS}@{_ONELAKE}/{_LAKEHOUSE_ID[lakehouse]}/Tables/{table_name}")
    return spark.read.format("delta").load(path)



SCHEMA_VERSION = "1.0"
summary_table = spark.read.table("gold_case_summary")
RUN_ID = PIPELINE_RUN_ID or (summary_table.orderBy(F.desc("generated_at_utc"))
                             .select("run_id").first()["run_id"])
GENERATED_AT = datetime.now(timezone.utc).isoformat()

cases = [row["case_id"] for row in
         summary_table.filter(F.col("run_id") == RUN_ID)
         .select("case_id").orderBy("case_id").collect()]
if CASE_ID:
    cases = [CASE_ID]
print("run_id:", RUN_ID, " cases:", cases)

In [ ]:
release = (lake_table("bronze_lakehouse", "bronze_reference_release")
           .filter(F.col("run_id") == RUN_ID).first())
assessments = spark.read.table("gold_case_variant_assessment").filter(F.col("run_id") == RUN_ID)
coverage = spark.read.table("gold_case_coverage").filter(F.col("run_id") == RUN_ID)
summaries = summary_table.filter(F.col("run_id") == RUN_ID)
quality = lake_table("silver_lakehouse", "silver_field_quality").filter(F.col("run_id") == RUN_ID)


def build_envelope(case_id):
    evidence = []

    def cite(kind, title, source_table, values):
        """Assign the next id and record what produced it."""
        identifier = f"EV-{len(evidence) + 1:03d}"
        evidence.append({
            "id": identifier, "kind": kind, "title": title,
            "sourceTable": source_table,
            "referenceRelease": release["read_at_utc"] if release else None,
            "values": values,
        })
        return identifier

    summary = summaries.filter(F.col("case_id") == case_id).first().asDict()
    cover = coverage.filter(F.col("case_id") == case_id).first().asDict()
    rows = [r.asDict() for r in
            assessments.filter(F.col("case_id") == case_id).collect()]

    release_id = cite(
        "referenceRelease", "Reference release read for this run",
        "bronze_reference_release",
        {"source": release["source"] if release else None,
         "readAtUtc": release["read_at_utc"] if release else None,
         "panel": release["panel_name"] if release else None,
         "genesReturned": release["genes_returned"] if release else None,
         "variantRecords": release["variant_records"] if release else None})

    coverage_id = cite(
        "coverage", f"Panel coverage for {case_id}", "gold_case_coverage",
        {"panelApplied": cover["panel_applied"],
         "genesCovered": cover["genes_covered"],
         "coverageState": cover["coverage_state"],
         "geneNotCovered": cover["gene_of_interest_not_covered"],
         "note": cover["coverage_note"]})

    variants = []
    for row in rows:
        identifier = cite(
            "variantAssessment",
            f"{row['accession']} ({row['assessment_state']})",
            "gold_case_variant_assessment",
            {"accession": row["accession"],
             "gene": row["gene_symbol"],
             "variantTitle": row["variant_title"],
             "proteinChange": row["protein_change"],
             "clinicalSignificance": row["clinical_significance"],
             "reviewStatus": row["review_status"],
             "reviewConfidence": row["review_confidence"],
             "conditions": row["condition_names"],
             "conditionIdentifiers": row["condition_identifiers"],
             "assessmentState": row["assessment_state"],
             "reportable": row["reportable"],
             "note": row["interpretation_note"]})
        variants.append({
            "accession": row["accession"],
            "gene": row["gene_symbol"],
            "clinicalSignificance": row["clinical_significance"],
            "assessmentState": row["assessment_state"],
            "reportable": row["reportable"],
            "evidenceId": identifier,
        })

    # Field-quality signal travels with the case. If a citable field was empty this run,
    # the agent is told rather than left to infer meaning from its absence.
    empty_fields = [row["column_name"] for row in
                    quality.filter(F.col("status") == "empty").collect()]
    quality_id = cite(
        "fieldQuality", "Citable fields empty in this run", "silver_field_quality",
        {"emptyFields": empty_fields, "count": len(empty_fields)})

    return {
        "schemaVersion": SCHEMA_VERSION,
        "run": {"runId": RUN_ID, "generatedAtUtc": GENERATED_AT,
                "referenceReleaseEvidenceId": release_id},
        "case": {"caseId": case_id,
                 "referralIndication": summary["referral_indication"],
                 "phenotypeTerms": summary["phenotype_terms"],
                 "panelApplied": summary["panel_applied"],
                 "highestTier": summary["highest_tier"],
                 "hasCoverageGap": summary["has_coverage_gap"],
                 "synthetic": True},
        "coverage": {"evidenceId": coverage_id,
                     "state": cover["coverage_state"],
                     "geneNotCovered": cover["gene_of_interest_not_covered"]},
        "variants": variants,
        "fieldQuality": {"evidenceId": quality_id, "emptyFields": empty_fields},
        "constraints": [
            "This envelope is the only permitted source of fact.",
            "A variant with assessmentState no_reference_entry is unclassified, not benign.",
            "A coverageState of gene_not_covered means no conclusion may be drawn about "
            "that gene.",
            "A variant of uncertain significance is not a negative result and must not be "
            "described as reassuring.",
            "No management, treatment or diagnostic recommendation may be made.",
        ],
        "evidence": evidence,
    }

In [ ]:
def validate(envelope):
    """Fail closed. A malformed envelope must never reach the agent."""
    problems = []
    declared = {record["id"] for record in envelope["evidence"]}

    if envelope["schemaVersion"] != SCHEMA_VERSION:
        problems.append("schemaVersion mismatch")
    if not envelope["case"]["caseId"]:
        problems.append("missing caseId")

    referenced = {envelope["run"]["referenceReleaseEvidenceId"],
                  envelope["coverage"]["evidenceId"],
                  envelope["fieldQuality"]["evidenceId"]}
    referenced |= {variant["evidenceId"] for variant in envelope["variants"]}
    dangling = {r for r in referenced if r} - declared
    if dangling:
        problems.append(f"evidence ids referenced but not declared: {sorted(dangling)}")

    # Reconcile against gold rather than trusting the envelope it just built.
    expected = assessments.filter(F.col("case_id") == envelope["case"]["caseId"]).count()
    if expected != len(envelope["variants"]):
        problems.append(f"variant count {len(envelope['variants'])} != gold {expected}")

    for variant in envelope["variants"]:
        if variant["assessmentState"] == "no_reference_entry" and variant["reportable"]:
            problems.append(f"{variant['accession']}: unclassified marked reportable")
        if variant["clinicalSignificance"] in ("Benign", "Likely benign") \
                and variant["reportable"]:
            problems.append(f"{variant['accession']}: benign marked reportable")
    return problems


WS = notebookutils.runtime.context["currentWorkspaceId"]
LH_ID = notebookutils.lakehouse.get("gold_lakehouse", workspaceId=WS).id
BASE = f"/lakehouse/default/Files/agent-handoff/{RUN_ID}"

published = []
for case_id in cases:
    envelope = build_envelope(case_id)

    problems = validate(envelope)
    if problems:
        raise ValueError(f"{case_id} quarantined:\n  " + "\n  ".join(problems))

    folder = f"{BASE}/{case_id}"
    os.makedirs(folder, exist_ok=True)
    payload = json.dumps(envelope, indent=1)
    with open(f"{folder}/case-evidence.json", "w", encoding="utf-8") as handle:
        handle.write(payload)

    # Re-read and re-validate what actually landed on disk.
    with open(f"{folder}/case-evidence.json", encoding="utf-8") as handle:
        written = json.load(handle)
    problems = validate(written)
    if problems:
        raise ValueError(f"{case_id} failed post-write validation:\n  "
                         + "\n  ".join(problems))

    with open(f"{folder}/_SUCCESS", "w", encoding="utf-8") as handle:
        handle.write(json.dumps({"runId": RUN_ID, "caseId": case_id,
                                 "evidenceRecords": len(envelope["evidence"]),
                                 "generatedAtUtc": GENERATED_AT}, indent=1))

    published.append({"caseId": case_id,
                      "evidence": len(envelope["evidence"]),
                      "variants": len(envelope["variants"]),
                      "coverage": envelope["coverage"]["state"],
                      "bytes": len(payload)})
    print(f"  {case_id}: {len(envelope['evidence'])} evidence records, "
          f"{len(payload):,} bytes")

print("\npublished:")
print(json.dumps(published, indent=1))

In [ ]:
preview = build_envelope(cases[0])
print(json.dumps({k: preview[k] for k in
                  ("schemaVersion", "run", "case", "coverage", "variants")}, indent=1))
print("\nfirst three evidence records:")
print(json.dumps(preview["evidence"][:3], indent=1))